# Smart Energy Model — prezentacja projektu dyplomowego

**Autor:** Marta Gałuszka  
**Data:** 2026-08-11  
**Repo:** smart-energy-model  

Ten notebook = **prezentacja**: w modelowaniu jest **kod treningu** (fit), nie tylko PNG. Pełny research/FE = notebooki 01–02.

**Architektura i pipeline:** Slajd **1b** (warstwy repo, notebooki 01–05) · pełny diagram flow: Slajd **9**.

**Demo kodu (Run All):** [`04_prezentacja_kod.ipynb`](04_prezentacja_kod.ipynb) · raport wyników: [`05_raport_wynikow.ipynb`](05_raport_wynikow.ipynb)

**Stan produkcji:** GPS · ICON · target PVE · primary **RF 16** · closeouty 14.07–**24.08** · retrening weekly **23.08**

**Linki do skryptów:** przy każdym slajdzie jest blok **Źródła / skrypty** — ścieżki względne z `notebooks/` (`../scripts/…`, `../mlops/…`).


In [1]:
# Konfiguracja ścieżek — URUCHOM NAJPIERW (albo Run All całego notebooka)
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image, display, Markdown

def _find_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, here.parent):
        if (cand / "src").is_dir() and (cand / "docs").is_dir():
            return cand
    if here.name == "notebooks":
        return here.parent
    if (here / "notebooks").is_dir():
        return here
    return here.parent

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

DOCS = ROOT / "docs"
FIGURES = ROOT / "reports" / "figures"
FIGURES_GH = ROOT / "docs" / "images" / "ml"
DATA = ROOT / "data" / "processed"
FORECASTS = DATA / "forecasts"

def show_table(df):
    """Jedna tabela Markdown — bez podwójnego HTML + text/plain z display(df)."""
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
    cols = [str(c) for c in d.columns]
    lines = [
        "| " + " | ".join(cols) + " |",
        "| " + " | ".join(["---"] * len(cols)) + " |",
    ]
    for _, row in d.iterrows():
        lines.append("| " + " | ".join(str(v) for v in row.tolist()) + " |")
    display(Markdown("\n".join(lines)))

def show_fig(name: str, width: int = 900):
    """PNG przez ścieżkę względną — render na GitHub bez base64 w .ipynb."""
    rel = f"../docs/images/ml/{name}"
    for base in (FIGURES_GH, FIGURES, DOCS):
        path = base / name
        if path.exists():
            display(Markdown(f'<img src="{rel}" width="{width}" alt="{name}"/>'))
            return
    print(f"⚠️ Brak pliku: {FIGURES_GH / name}")

print(f"ROOT: {ROOT}")
print(f"FIGURES: {FIGURES}")
_needed = [
    "july_validation_plot.png",
    "production_validation_plot.png",
    "production_validation.png",
    "ablation_chart.png",
    "calendar_ablation_comparison.png",
    "learning_curves.png",
    "hourly_algorithm_scatter.png",
    "production_learning_curves.png",
    "prediction_vs_actual_train_vs_holdout.png",
]
missing = [n for n in _needed if not any((b / n).exists() for b in (FIGURES, FIGURES_GH, DOCS))]
if missing:
    print("⚠️ Brak PNG:", ", ".join(missing))
    print("Regeneracja: MPLBACKEND=Agg PYTHONPATH=$PWD python scripts/plots/plot_july_validation.py")
    print("             MPLBACKEND=Agg PYTHONPATH=$PWD python scripts/plots/plot_production_validation.py")
else:
    print("✓ Wykresy dostępne (reports/figures lub docs/images/ml)")


ROOT: .
FIGURES: ./reports/figures
✓ Wykresy dostępne (reports/figures lub docs/images/ml)


---
## Slajd 1 — Tytuł

**Smart Energy Model**  
Prognoza produkcji fotowoltaicznej i wsparcie sterowania magazynem energii

Instalacja domowa · FoxESS · Open-Meteo · Random Forest · MLOps

**Źródła / skrypty:**
- repo / README: [`README.md`](../README.md)
- założenia: [`docs/03_ZALOZENIA_I_DECYZJE.md`](../docs/03_ZALOZENIA_I_DECYZJE.md)
- mapa skryptów: [`scripts/README.md`](../scripts/README.md) · [`mlops/README.md`](../mlops/README.md)


---
## Slajd 1b — Architektura repozytorium i pipeline

Projekt to **warstwy produkcyjne** — każdy katalog ma jedną rolę; logika ML nie jest kopiowana między notebookami.

### Warstwy kodu

| Warstwa | Katalog | Rola |
|---------|---------|------|
| Biblioteki | [`src/`](../src/) | dane, cechy, model, finanse, optymalizacja |
| Trening / analizy | [`scripts/`](../scripts/) | `train/`, `analysis/`, `plots/` |
| Produkcja (host) | [`mlops/`](../mlops/) | sync, prognoza, closeout, launchd |
| API | [`api/`](../api/) | FastAPI, Pydantic, JWT |
| Aplikacja | [`mobile/`](../mobile/) | Ionic / Capacitor |
| Artefakty | [`models/`](../models/) | `.joblib` + `.metadata.json` |

### Notebooki — podział ról

| Notebook | Rola |
|----------|------|
| [`01_EDA_analiza_danych.ipynb`](01_EDA_analiza_danych.ipynb) | EDA, jakość danych IoT, korelacje |
| [`02_ML_predykcja_PV.ipynb`](02_ML_predykcja_PV.ipynb) | feature engineering, ablacja 19→16, pełny research ML |
| **Ten plik (`03`)** | slajdy prezentacji (obrona) |
| [`04_prezentacja_kod.ipynb`](04_prezentacja_kod.ipynb) | liniowy demo: dane → cech → `.fit()` → MAPE (Run All) |
| [`05_raport_wynikow.ipynb`](05_raport_wynikow.ipynb) | raport Markdown z CSV → [`reports/model_comparison.md`](../reports/model_comparison.md) |

### Pipeline end-to-end (dane → API)

| Etap | Komponent | Plik |
|------|-----------|------|
| 1 Sync danych | FoxESS + Open-Meteo → SQLite | [`mlops/sync_data.py`](../mlops/sync_data.py) |
| 2 Feature engineering | 16 cech produkcyjnych | [`src/features/pv_features_hourly_extended.py`](../src/features/pv_features_hourly_extended.py) |
| 3 Trening | GridSearch RF, MLflow, artefakt | [`scripts/train/train_hourly_model_tuning.py`](../scripts/train/train_hourly_model_tuning.py) |
| 4 Inferencja | prognoza 1–3 dni | [`mlops/forecast_pv.py`](../mlops/forecast_pv.py) |
| 5 API | prognoza godzinowa | [`api/main.py`](../api/main.py) → `GET /api/v1/forecast/hourly` |
| 6 Kontener | Docker + Postgres | [`Dockerfile`](../Dockerfile) · [`docker-compose.yml`](../docker-compose.yml) |
| 7 Walidacja live | closeout vs FoxESS | [`mlops/evening_closeout.py`](../mlops/evening_closeout.py) |

Diagram ASCII (pełny flow): **Slajd 9**. Dokumentacja: [`README.md`](../README.md) · metryki: [`docs/STATUS_ML_MLOPS.md`](../docs/STATUS_ML_MLOPS.md)

### Metryki (regresja PV, kWh/h)

- **Offline:** Test MAE, R², gap train–test (holdout 80/20 po dniach)
- **Live:** MAPE dzienne vs raport FoxESS (`forecast_validation.csv`)
- **Artefakt:** Random Forest 16 cech · target Δ`PVEnergyTotal`

**Źródła:** [`docs/PLAN_DYPLOM_CHECKLIST.md`](../docs/PLAN_DYPLOM_CHECKLIST.md) · [`scripts/README.md`](../scripts/README.md) · [`mlops/README.md`](../mlops/README.md)

---
## Slajd 2 — Problem

**Pytanie biznesowe:** Kiedy dom produkuje najwięcej PV i ile energii będzie dostępne jutro / dziś?

**Cel:** Godzinowa prognoza produkcji (kWh/h) → harmonogram AGD, rekomendacje baterii (taryfa G12w).

**Metryka sukcesu:**
- offline: Test MAE ≤ **0,70 kWh/h**, gap train–test niski
- operacyjnie: MAE prognozy dziennej vs FoxESS (rolling 7–14 dni)

**Ograniczenia:** ~14 miesięcy danych, prognoza pogody (nie archiwum), brak auto-apply na falownik (dry-run).

**Źródła / skrypty:** problem biznesowy / założenia — [`docs/03_ZALOZENIA_I_DECYZJE.md`](../docs/03_ZALOZENIA_I_DECYZJE.md)


---
## Slajd 3 — Dane

| Źródło | Co daje | Rozdzielczość |
|--------|---------|---------------|
| **FoxESS API** | PV, bateria, load; **target ML = `PVEnergyTotal`** (licznik jak w app) | ~5 min → godziny (Δ licznika) |
| **Open-Meteo** | temp, wilgotność, chmury, radiacja, wiatr (`icon_seamless`) | godzinowa |
| **Tauron** | rachunki / eksport — **tylko ROI**, nie w treningu | miesięczna |

**Target:** `pv_kwh_hour` = dodatnie delty `PVEnergyTotal` (`PV_HOURLY_TARGET=pve`).  
Trening i closeout używają **tej samej** zmiennej — bez skalowania `pvPower`.

**Źródła / skrypty:**
- sync FoxESS + pogoda → SQLite: [`mlops/sync_data.py`](../mlops/sync_data.py)
- Open-Meteo (ICON): [`scripts/analysis/fetch_weather.py`](../scripts/analysis/fetch_weather.py)
- target PVE / cechy godzinowe: [`src/features/pv_features_hourly_extended.py`](../src/features/pv_features_hourly_extended.py)
- docs target: [`docs/UPDATE_2026-07-18_target-pve.md`](../docs/UPDATE_2026-07-18_target-pve.md)


In [2]:
# Źródło zakresu: SQLite po mlops/sync_data.py
import sqlite3

db = ROOT / 'data' / 'energy_model.db'
if db.exists():
    conn = sqlite3.connect(db)
    days = pd.read_sql(
        "SELECT MIN(date(timestamp)) AS od, MAX(date(timestamp)) AS do, COUNT(DISTINCT date(timestamp)) AS dni FROM foxess_data",
        conn,
    )
    conn.close()
    print('=== Zakres danych FoxESS ===')
    print(days.to_string(index=False))
else:
    print('⚠️ Brak bazy — uruchom sync_data.py')


=== Zakres danych FoxESS ===
        od         do  dni
2025-04-25 2026-08-23  471


---
## Slajd 4 — EDA (odkrycia)

- Sezonowość PV (lato vs zima) — duża różnica produkcji
- Pochmurne / burzowe dni → największy rozrzut prognoza vs rzeczywistość
- **Start potoku MLOps (launchd)** — wg `cron.log`:

| Data | Co wystartowało |
|------|-----------------|
| **14.07** | **midday 12:00** (pierwszy live) |
| **15.07** | regularne **daily 5:00** + evening closeout |
| **16.07** | **peak 16:00** |

Szczegółowa ewaluacja live (raw vs hybryda vs app) — **po wyborze modelu**, slajd 8b.

**Źródła / skrypty:**
- EDA (kod): [`notebooks/01_EDA_analiza_danych.ipynb`](01_EDA_analiza_danych.ipynb) · [`docs/01_EDA_analiza.md`](../docs/01_EDA_analiza.md)
- start MLOps / logi: [`mlops/daily_workflow.sh`](../mlops/daily_workflow.sh), [`mlops/midday_forecast.sh`](../mlops/midday_forecast.sh), [`mlops/peak_arrival.sh`](../mlops/peak_arrival.sh), [`mlops/evening_closeout.sh`](../mlops/evening_closeout.sh)


In [3]:
# (slajd 4 — bez osobnych wykresów; case study live jest na slajdzie 8b)
print('EDA: sezonowość + start MLOps — szczegóły live później')


EDA: sezonowość + start MLOps — szczegóły live później


---
## Slajd 5 — Feature engineering

**16 cech produkcyjnych** (`HOURLY_FEATURE_COLUMNS_PRODUCTION`):
- pogoda: temp, wilgotność, chmury, radiacja, wiatr
- słońce: wschód/zachód, `sun_position`, `is_daylight`
- reguły: śnieg na panelach, mgła

**Ablacja:** wyrzucono `month`, `doy_*` (redundantne wobec radiacji + słońca).

Wykresy w komórce poniżej: ablacja → kalendarz vs słońce → krzywe uczenia etapów.

**Źródła / skrypty:**
- definicja 16 cech: [`src/features/pv_features_hourly_extended.py`](../src/features/pv_features_hourly_extended.py) (`HOURLY_FEATURE_COLUMNS_PRODUCTION`)
- ablacja (CSV): [`scripts/analysis/ablation_study.py`](../scripts/analysis/ablation_study.py)
- wykres ablacji: [`scripts/plots/plot_error_chart.py`](../scripts/plots/plot_error_chart.py) → `ablation_chart.png`
- kalendarz vs słońce: [`scripts/plots/plot_calendar_ablation.py`](../scripts/plots/plot_calendar_ablation.py) → `calendar_ablation_comparison.png`
- krzywe etapów: [`scripts/plots/plot_learning_curves.py`](../scripts/plots/plot_learning_curves.py) → `learning_curves.png`


In [4]:
# CSV/PNG: scripts/analysis/ablation_study.py + scripts/plots/plot_{error_chart,calendar_ablation,learning_curves}.py
abl = pd.read_csv(DATA / 'ablation_results.csv')
print('=== Ablacja cech (Test MAE) ===')
show_table(abl[['Etap', 'N_cech', 'Test_MAE', 'Test_R2']].round(3))
show_fig('ablation_chart.png', width=700)

cal_path = DATA / 'calendar_ablation_comparison.csv'
if cal_path.exists():
    print('\n=== Kalendarz vs Pogoda+Słońce ===')
    show_table(pd.read_csv(cal_path).round(3))
show_fig('calendar_ablation_comparison.png', width=800)

show_fig('learning_curves.png', width=800)


=== Ablacja cech (Test MAE) ===


| Etap | N_cech | Test_MAE | Test_R2 |
| --- | --- | --- | --- |
| 1_Baza | 1 | 1.072 | 0.258 |
| 2_Pogoda | 6 | 0.622 | 0.647 |
| 3_Kalendarz | 9 | 0.605 | 0.672 |
| 3_Pogoda_Slonce | 13 | 0.580 | 0.691 |
| 3_Pogoda_Slonce_Reguly | 16 | 0.581 | 0.690 |
| 4_Reguly | 19 | 0.578 | 0.693 |

<img src="../docs/images/ml/ablation_chart.png" width="700" alt="ablation_chart.png"/>


=== Kalendarz vs Pogoda+Słońce ===


| Etap | N_cech | Test_MAE | Test_RMSE | Test_R2 | Delta_MAE_vs_Pogoda | Delta_MAE_vs_Pelny | Wdrozony | Rekomendowany |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2_Pogoda | 6 | 0.622 | 0.887 | 0.647 | 0.000 | 0.044 | False | False |
| 3_Kalendarz | 9 | 0.605 | 0.856 | 0.672 | -0.017 | 0.027 | False | False |
| 3_Pogoda_Slonce | 13 | 0.580 | 0.831 | 0.691 | -0.042 | 0.002 | False | False |
| 3_Pogoda_Slonce_Reguly | 16 | 0.581 | 0.831 | 0.690 | -0.041 | 0.003 | True | True |
| 4_Reguly | 19 | 0.578 | 0.827 | 0.693 | -0.044 | 0.000 | False | False |

<img src="../docs/images/ml/calendar_ablation_comparison.png" width="800" alt="calendar_ablation_comparison.png"/>

<img src="../docs/images/ml/learning_curves.png" width="800" alt="learning_curves.png"/>

---
## Slajd 6 — Podejście do modelowania

- **Target:** Δ`PVEnergyTotal` · **cechy:** 16 · **split:** 80/20 po dniach
- **Kandydaci (ten sam setup):** Ridge → Random Forest → XGBoost
- **Wybór:** Test MAE **+** gap (przeuczenie)
- **Następny slajd:** komórka z **kodem** `.fit()` dla trzech modeli (nie sam odczyt CSV)

**Źródła / skrypty:**
- split 80/20 + wizualizacja: [`scripts/train/train_hourly_model_tuning.py`](../scripts/train/train_hourly_model_tuning.py) → `data_split_viz.png`
- protokół porównania algorytmów: [`scripts/analysis/compare_algorithms_hourly.py`](../scripts/analysis/compare_algorithms_hourly.py)


In [5]:
# PNG: scripts/train/train_hourly_model_tuning.py → ../docs/images/ml/data_split_viz.png
show_fig('data_split_viz.png', width=900)


<img src="../docs/images/ml/data_split_viz.png" width="900" alt="data_split_viz.png"/>

---
## Slajd 7 — Kod: porównanie Ridge / RF / XGBoost

Poniżej **uruchamialny kod** (ten sam protokół co `scripts/analysis/compare_algorithms_hourly.py`):

1. wczytaj ramkę godzinową (16 cech, PVE)  
2. split 80/20 **po dniach**  
3. `fit` Ridge, RF (parametry prod.), XGBoost  
4. metryki Test MAE / R² / gap → werdykt  

Potem wykresy. **Slajd 8** = tylko wybrany RF (offline). **8b** = live.

**Źródła / skrypty:**
- **obliczenia w komórce poniżej** = ten sam protokół co [`scripts/analysis/compare_algorithms_hourly.py`](../scripts/analysis/compare_algorithms_hourly.py)
- wykresy MAE/R²/scatter: ten skrypt → `hourly_algorithm_errors_mae_rmse.png`, `hourly_algorithm_r2.png`, `hourly_algorithm_scatter.png`
- CSV wyników: `data/processed/hourly_algorithm_comparison.csv`


In [6]:
# === KOD MODELOWANIA — fit 3 algorytmów ===
# Ten sam protokół co scripts/analysis/compare_algorithms_hourly.py
import os
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

def _gap_verdict(gap: float) -> str:
    if gap < 0.15:
        return '✅ Nie przeuczony'
    if gap < 0.35:
        return '⚠️ Lekkie przeuczenie'
    return '❌ Przeuczony'

cmp = None
try:
    from xgboost import XGBRegressor
    from src.features.pv_features_hourly_extended import (
        HOURLY_FEATURE_COLUMNS_PRODUCTION,
        TARGET_COLUMN,
        load_hourly_training_frame_extended,
    )
    from src.models.pv_hourly_predictor import (
        RF_MAX_DEPTH, RF_MAX_FEATURES, RF_MIN_SAMPLES_LEAF,
        RF_MIN_SAMPLES_SPLIT, RF_N_ESTIMATORS, RF_RANDOM_STATE,
    )

    TRAIN_START, TRAIN_END = '2025-06-01', '2026-05-31'
    lat = float(os.getenv('WEATHER_LAT', '50.06'))
    lon = float(os.getenv('WEATHER_LON', '19.94'))

    print('1) Dane (16 cech, PVE, split 80/20 po dniach)…')
    frame = load_hourly_training_frame_extended(latitude=lat, longitude=lon)
    frame = frame[(frame['day'] >= TRAIN_START) & (frame['day'] <= TRAIN_END)].copy()
    days = frame['day'].unique()
    days_train, days_test = train_test_split(days, test_size=0.2, random_state=42, shuffle=True)
    tr = frame['day'].isin(days_train)
    te = frame['day'].isin(days_test)
    feats = list(HOURLY_FEATURE_COLUMNS_PRODUCTION)
    X_tr = frame.loc[tr, feats].replace([np.inf, -np.inf], np.nan)
    y_tr = frame.loc[tr, TARGET_COLUMN]
    X_te = frame.loc[te, feats].replace([np.inf, -np.inf], np.nan)
    y_te = frame.loc[te, TARGET_COLUMN]
    print(f'   {len(frame)} h / {frame["day"].nunique()} dni | train {tr.sum()} h | test {te.sum()} h | target={TARGET_COLUMN}')

    def eval_model(name, pipe):
        pipe.fit(X_tr, y_tr)
        mae_tr = mean_absolute_error(y_tr, pipe.predict(X_tr))
        mae_te = mean_absolute_error(y_te, pipe.predict(X_te))
        gap = mae_te - mae_tr
        return {
            'label': name,
            'test_mae_hour': mae_te,
            'test_r2_hour': r2_score(y_te, pipe.predict(X_te)),
            'gap_hour': gap,
            'verdict': _gap_verdict(gap),
        }

    models = {
        'Ridge': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', Ridge(alpha=100.0, random_state=42)),
        ]),
        'Random Forest': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', RandomForestRegressor(
                n_estimators=RF_N_ESTIMATORS,
                max_depth=RF_MAX_DEPTH,
                min_samples_leaf=RF_MIN_SAMPLES_LEAF,
                min_samples_split=RF_MIN_SAMPLES_SPLIT,
                max_features=RF_MAX_FEATURES,
                random_state=RF_RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
        'XGBoost': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', XGBRegressor(
                n_estimators=200, max_depth=6, learning_rate=0.1,
                subsample=0.9, colsample_bytree=0.9, random_state=42,
                objective='reg:squarederror', n_jobs=-1,
            )),
        ]),
    }

    print('\n2) fit Ridge → RF → XGBoost…')
    rows = []
    for name, pipe in models.items():
        print(f'   {name}…', end=' ', flush=True)
        rows.append(eval_model(name, pipe))
        print('OK')
    cmp = pd.DataFrame(rows)
    print('\n3) Wyniki (live fit w tej komórce):')
except Exception as e:
    print(f'⚠️ Live fit niedostępny ({type(e).__name__}: {e})')
    print('   → CSV z tego samego protokołu. Kod .fit() zostaje powyżej do oceny.')
    cmp = pd.read_csv(DATA / 'hourly_algorithm_comparison.csv')
    cols = ['label', 'test_mae_hour', 'test_r2_hour', 'gap_hour', 'verdict']
    cmp = cmp[[c for c in cols if c in cmp.columns]]
    print('3) Wyniki (hourly_algorithm_comparison.csv):')

show_table(cmp.round(3))
print('\n→ Wybór: Random Forest (MAE blisko XGB, dużo mniejszy gap).')
print('\n4) Wykresy:')
show_fig('hourly_algorithm_errors_mae_rmse.png')
show_fig('hourly_algorithm_r2.png')
show_fig('hourly_algorithm_scatter.png')


FoxESS-Cloud Open API version 2.9.15


./venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


1) Dane (16 cech, PVE, split 80/20 po dniach)…
✓ Target godzinowy = PVEnergyTotal (Δ licznika, jak w app): 439 dni, 4951 godzin
✓ Dodano flagi śniegu z modelu topnienia (dni ze śniegiem: 35 / 439)
✓ Dodano flagę mgły (dni z mgłą: 146 / 439)
📊 Statystyki godzin produkcji:
   Najwcześniejsza: 5:00
   Najpóźniejsza: 20:00
   Średni wschód słońca: 5.59
   Średni zachód słońca: 19.32
   3505 h / 356 dni | train 2779 h | test 726 h | target=pv_kwh_hour

2) fit Ridge → RF → XGBoost…
   Ridge… OK
   Random Forest… 

./venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
./venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
./venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
./venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
./venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
./venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
./venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ co

OK
   XGBoost… OK

3) Wyniki (live fit w tej komórce):


| label | test_mae_hour | test_r2_hour | gap_hour | verdict |
| --- | --- | --- | --- | --- |
| Ridge | 0.831 | 0.526 | -0.003 | ✅ Nie przeuczony |
| Random Forest | 0.602 | 0.675 | 0.096 | ✅ Nie przeuczony |
| XGBoost | 0.614 | 0.654 | 0.470 | ❌ Przeuczony |


→ Wybór: Random Forest (MAE blisko XGB, dużo mniejszy gap).

4) Wykresy:


<img src="../docs/images/ml/hourly_algorithm_errors_mae_rmse.png" width="900" alt="hourly_algorithm_errors_mae_rmse.png"/>

<img src="../docs/images/ml/hourly_algorithm_r2.png" width="900" alt="hourly_algorithm_r2.png"/>

<img src="../docs/images/ml/hourly_algorithm_scatter.png" width="900" alt="hourly_algorithm_scatter.png"/>

---
## Slajd 8 — Wybrany model: Random Forest (16 cech)

Po porównaniu z slajdu 7: **wdrażamy RF**, nie Ridge i nie XGBoost.

**Artefakt:** `models/pv_hourly_model.joblib` · GPS · ICON · target **PVE**  

Poniżej ewaluacja **offline** (metryki expanding + krzywe + szereg TRAIN|HOLDOUT).  
Live closeouty → **slajd 8b**.

**Źródła / skrypty:**
- trening / GridSearch RF (prod): [`scripts/train/train_hourly_model_tuning.py`](../scripts/train/train_hourly_model_tuning.py) → `hourly_model_tuning_summary_production.csv`
- krzywe uczenia RF: [`scripts/plots/plot_rf_convergence.py`](../scripts/plots/plot_rf_convergence.py) → `production_learning_curves.png`
- szereg TRAIN|HOLDOUT: [`scripts/plots/plot_pv_timeseries_comparison.py`](../scripts/plots/plot_pv_timeseries_comparison.py) → `prediction_vs_actual_train_vs_holdout.png`
- predyktor: [`src/models/pv_hourly_predictor.py`](../src/models/pv_hourly_predictor.py)


In [7]:
# CSV/PNG: train_hourly_model_tuning.py · plot_rf_convergence.py · plot_pv_timeseries_comparison.py
# Wybrany model — ewaluacja OFFLINE
prod = pd.read_csv(DATA / 'hourly_model_tuning_summary_production.csv')
print('=== Model produkcyjny RF 16 (expanding window) ===')
show_table(prod[['train_start', 'train_end', 'test_mae_hour', 'gap', 'daily_mae', 'verdict']].round(3))

show_fig('production_learning_curves.png', width=700)
print('\n=== Szereg: ★ RF vs rzeczywistość (TRAIN | HOLDOUT) ===')
show_fig('prediction_vs_actual_train_vs_holdout.png', width=1000)


=== Model produkcyjny RF 16 (expanding window) ===


| train_start | train_end | test_mae_hour | gap | daily_mae | verdict |
| --- | --- | --- | --- | --- | --- |
| 2025-06-01 | 2026-08-22 | 0.658 | 0.069 | 3.666 | ✅ Model NIE jest przeuczony |

<img src="../docs/images/ml/production_learning_curves.png" width="700" alt="production_learning_curves.png"/>


=== Szereg: ★ RF vs rzeczywistość (TRAIN | HOLDOUT) ===


<img src="../docs/images/ml/prediction_vs_actual_train_vs_holdout.png" width="1000" alt="prediction_vs_actual_train_vs_holdout.png"/>

---
## Slajd 8b — Ewaluacja live wybranego RF (closeouty)

Jak RF radzi sobie **na produkcji** (14.07→24.08) — po wyborze modelu.

- **Raw** = sam RF na cały dzień · **Hybryda dnia (CS4)** = FoxESS (minione) + RF (przyszłe) — **nie** ADJUST  
- Actual = Δ`PVEnergyTotal` (= app)  
- MAPE po **zmianach logiki** (PVE / dual), nie po każdym weekly odświeżeniu wag  

**Era dual 27.07–24.08:** MAPE raw 5:00 ~**11,0%** · 12:00 ~**10,0%** (n=29).  
Całość 14.07–24.08: ~**15,4%** / **13,5%** (42 closeoutów).

**Case study 09.08 (słońce):** actual **37,7 kWh** · raw 5:00 **33,84** (APE **10,2%**) · **CS4 33,47** (APE **11,2%**) — raw zaniża po retreningu 09.08.  
**Case study 10.08 (pochmurny):** actual **28,9 kWh** · raw 5:00 **30,57** (APE **5,8%**) · **CS4 29,63** (APE **2,5%**) — CS4 bliżej app.  
**Case study 07.08 (burzy):** actual **13,6 kWh** · **CS4 13,81** (APE **1,5%**) — shadow ratuje słaby dzień.

**Shadow vs primary (RF 16) — dlaczego zostaje 16:**

| Porównanie | CS4 shadow lepiej | Primary lepiej | Uwaga |
|------------|-------------------|------------------|--------|
| **Live closeout** daily |err| (27.07–24.08) | **CS4 lepszy 9 / 29** | RF częściej | m.in. **07–08**, **10**, **16–17.08**; **19**, **23–24.08** zaniżenie RF; **20.08** best `peak_cs4`; **21.08** overshoot raw +32,9%; **22.08** best peak/xgb (~25,2 vs 25,1) |
| **Live closeout** midday 12:00 | **2 / 15** (13%) | 13 / 15 | m.in. **08.08** (XGB+TS midday najlepszy) |
| **Oneshot offline** (19–25.07) | **5 / 7** (71%) | 2 / 7 | obiecująco przed wdrożeniem — live inaczej |
| **XGB+TS shadow** (log 26.07–01.08) | **1 / 3** | 2 / 3 | za mało dni w closeoucie; wygrany **31.07** |

> **SHAP** = interpretacja cech (zachmurzenie, słońce) — **nie** model prognozy kWh. Patrz notebook 07.

**Źródła / skrypty:**
- closeout wieczorny (actual vs prognoza): [`mlops/evening_closeout.py`](../mlops/evening_closeout.py) → `forecast_validation.csv`
- porównanie dnia: [`scripts/analysis/compare_day_forecasts.py`](../scripts/analysis/compare_day_forecasts.py)
- wykres live raw/hybryda: [`scripts/plots/plot_production_validation.py`](../scripts/plots/plot_production_validation.py)
- wykres lipca + MAPE: [`scripts/plots/plot_july_validation.py`](../scripts/plots/plot_july_validation.py)
- prognoza hybrydowa: [`mlops/forecast_pv.py`](../mlops/forecast_pv.py)


In [8]:
# PNG/MD: plot_production_validation.py · plot_july_validation.py ← evening_closeout.py
val = pd.read_csv(DATA / 'forecasts' / 'forecast_validation.csv')
val['target_day'] = pd.to_datetime(val['target_day'])
end_label = val['target_day'].max().strftime('%d.%m.%Y')
print(f'=== Live: raw RF vs hybryda vs app (14.07→{end_label}) ===')
show_fig('production_validation_plot.png')
show_fig('july_validation_plot.png')
show_fig('production_validation.png', width=1000)

from scripts.plots.plot_july_validation import (
    load_july_validation, build_july_error_summary, DEFAULT_VALIDATION,
)
summary_path = FIGURES / 'july_validation_summary.md'
summary_path.write_text(
    build_july_error_summary(
        load_july_validation(DEFAULT_VALIDATION),
        db_path=ROOT / 'data' / 'energy_model.db',
    ),
    encoding='utf-8',
)
display(Markdown(summary_path.read_text(encoding='utf-8')))

CASE_DAYS = ['2026-08-09', '2026-08-10']

def _shadow_from_history(day, labels):
    hist_path = FORECASTS / 'forecast_history.csv'
    if not hist_path.exists():
        return []
    hist = pd.read_csv(hist_path)
    h = hist[hist['target_day'].astype(str) == day]
    out = []
    for name, run_label in labels:
        snap = h[h['run_label'] == run_label].sort_values('run_at').tail(1)
        if not snap.empty:
            out.append((name, float(snap.iloc[0]['predicted_kwh'])))
    return out

for CLOSEOUT_DAY in CASE_DAYS:
    if CLOSEOUT_DAY not in val['target_day'].dt.strftime('%Y-%m-%d').values:
        continue
    today = val[val['target_day'] == CLOSEOUT_DAY].iloc[0]
    act = float(today['actual_pv_report'])
    closeout_rows = [
        ('Actual (FoxESS app)', act),
        ('Primary RF 5:00', today['predicted_daily_raw']),
        ('Primary RF 12:00', today['predicted_midday_raw']),
        ('Shadow CS4 5:00', today.get('predicted_daily_cs4')),
        ('Shadow CS4 12:00', today.get('predicted_midday_cs4')),
    ]
    closeout_rows += _shadow_from_history(CLOSEOUT_DAY, [
        ('Shadow XGB+TS 5:00', 'daily_xgb_ts'),
        ('Shadow XGB+TS 12:00', 'midday_xgb_ts'),
    ])
    today_tbl = pd.DataFrame([
        {
            'wariant': n,
            'kWh': v,
            'APE_%': 0.0 if n.startswith('Actual') else abs(v - act) / act * 100,
        }
        for n, v in closeout_rows if pd.notna(v)
    ])
    print(f'\n=== {CLOSEOUT_DAY} — closeout (primary + shadow) ===')
    show_table(today_tbl.round(2))
    best = today_tbl.loc[today_tbl['wariant'] != 'Actual (FoxESS app)', 'APE_%'].idxmin()
    print(f'Najlepszy wariant dnia: {today_tbl.loc[best, "wariant"]} (APE {today_tbl.loc[best, "APE_%"]:.1f}%)')

# Shadow CS4 / XGB+TS vs primary RF 16 (live + oneshot)
dual = val[val['target_day'] >= '2026-07-27'].dropna(
    subset=['predicted_daily_cs4', 'predicted_daily_raw', 'actual_pv_report']
)
dual = dual[dual['actual_pv_report'] > 0.5].copy()
dual['err16'] = (dual['predicted_daily_raw'] - dual['actual_pv_report']).abs()
dual['err_cs4'] = (dual['predicted_daily_cs4'] - dual['actual_pv_report']).abs()
dual['cs4_lepiej'] = dual['err_cs4'] < dual['err16']
n_dual = len(dual)
n_cs4_win = int(dual['cs4_lepiej'].sum())

mid = dual.dropna(subset=['predicted_midday_cs4', 'predicted_midday_raw'])
mid['err16_m'] = (mid['predicted_midday_raw'] - mid['actual_pv_report']).abs()
mid['err_cs4_m'] = (mid['predicted_midday_cs4'] - mid['actual_pv_report']).abs()
n_mid_win = int((mid['err_cs4_m'] < mid['err16_m']).sum())

era_end = dual['target_day'].max().strftime('%d.%m')
shadow_rows = [
    {
        'porównanie': f'Live daily 5:00 (27.07→{era_end})',
        'shadow_lepiej': f'{n_cs4_win}/{n_dual}',
        'primary_lepiej': f'{n_dual - n_cs4_win}/{n_dual}',
        'pct_shadow': round(100 * n_cs4_win / n_dual, 1) if n_dual else None,
    },
    {
        'porównanie': 'Live midday 12:00',
        'shadow_lepiej': f'{n_mid_win}/{len(mid)}',
        'primary_lepiej': f'{len(mid) - n_mid_win}/{len(mid)}',
        'pct_shadow': round(100 * n_mid_win / len(mid), 1) if len(mid) else None,
    },
]

ldw_path = DATA / 'live_dual_week_20260726_0801.csv'
if ldw_path.exists():
    ldw = pd.read_csv(ldw_path)
    vx = ldw.dropna(subset=['pred_xgb_daily', 'pred_16_daily', 'actual'])
    if not vx.empty:
        xgb_win = int((
            (vx['pred_xgb_daily'] - vx['actual']).abs()
            < (vx['pred_16_daily'] - vx['actual']).abs()
        ).sum())
        shadow_rows.append({
            'porównanie': 'XGB+TS daily (log 26.07–01.08)',
            'shadow_lepiej': f'{xgb_win}/{len(vx)}',
            'primary_lepiej': f'{len(vx) - xgb_win}/{len(vx)}',
            'pct_shadow': round(100 * xgb_win / len(vx), 1),
        })

osw_path = DATA / 'oneshot_cs4_week_20260719_25.csv'
if osw_path.exists():
    osw = pd.read_csv(osw_path)
    osw['cs4_lepiej'] = osw['err_CS4'].abs() < osw['err_A baseline'].abs()
    n_os = len(osw)
    n_os_win = int(osw['cs4_lepiej'].sum())
    shadow_rows.append({
        'porównanie': 'Oneshot CS4 offline (19–25.07)',
        'shadow_lepiej': f'{n_os_win}/{n_os}',
        'primary_lepiej': f'{n_os - n_os_win}/{n_os}',
        'pct_shadow': round(100 * n_os_win / n_os, 1),
    })

print('\n=== Shadow vs primary RF 16 ===')
show_table(pd.DataFrame(shadow_rows))

if n_cs4_win:
    win_days = dual.loc[dual['cs4_lepiej'], 'target_day'].dt.strftime('%d.%m').tolist()
    print('CS4 daily wygrał:', ', '.join(win_days))
print('Wniosek: primary RF 16 zostaje — shadow tylko obserwacja (launchd).')


=== Live: raw RF vs hybryda vs app (14.07→24.08.2026) ===


<img src="../docs/images/ml/production_validation_plot.png" width="900" alt="production_validation_plot.png"/>

<img src="../docs/images/ml/july_validation_plot.png" width="900" alt="july_validation_plot.png"/>

<img src="../docs/images/ml/production_validation.png" width="1000" alt="production_validation.png"/>

### Podsumowanie błędów (lipiec) — pogoda i hybryda

Metryka: **|APE| %** = `|actual − prognoza| / actual × 100`. Pogoda: średnie **cloud ICON 6–20** z `weather_data`.

#### Kiedy błąd jest mniejszy / większy

| Typ dnia (cloud) | n | MAPE raw 5:00 | MAPE raw 12:00 | Dni |
|---|---:|---:|---:|---|
| słoneczny / mało chmur | 11 | 6.2% | 5.8% | 20.07, 26.07, 30.07, 04.08, 05.08, 06.08, 09.08, 12.08, 13.08, 14.08, 15.08 |
| mieszany | 13 | 12.8% | 13.8% | 16.07, 17.07, 25.07, 28.07, 29.07, 31.07, 03.08, 08.08, 16.08, 20.08, 22.08, 23.08, 24.08 |
| pochmurny / deszczowy | 18 | 22.8% | 18.1% | 14.07, 15.07, 18.07, 19.07, 21.07, 22.07, 23.07, 24.07, 27.07, 01.08, 02.08, 07.08, 10.08, 11.08, 17.08, 18.08, 19.08, 21.08 |

- **Najtrafniejszy raw 5:00:** 30.07 (0.6% · słoneczny / mało chmur, cloud~31%, actual 33.5 kWh), 03.08 (0.7% · mieszany, cloud~48%, actual 33.5 kWh), 05.08 (1.2% · słoneczny / mało chmur, cloud~9%, actual 34.9 kWh)
- **Najgorszy raw 5:00:** 24.07 (100.7% · pochmurny / deszczowy, cloud~86%, actual 10.7 kWh), 15.07 (59.5% · pochmurny / deszczowy, cloud~94%, actual 10.9 kWh), 21.07 (46.5% · pochmurny / deszczowy, cloud~85%, actual 18.8 kWh)

- **Wzorzec:** na dniach **jasnych / wysokiej produkcji** raw bywa lekko **za niski** (ICON za chmurny vs Accu) — błąd umiarkowany w %, duży w kWh. Na dniach **słabych / burzowych** raw często **zawyża** — wtedy |APE| % bywa największy.

#### Kiedy hybryda dnia pomaga, a kiedy szkodzi

Porównanie **midday (12:00)**: hybryda = FoxESS na minione godziny + RF na resztę. „Pomaga/szkodzi” = różnica |APE| raw−hybryda ≥ **1 pp**.

- **Hybryda pomaga (12:00):** 18.07 (↓14.1 pp, pochmurny / deszczowy); 21.07 (↓18.9 pp, pochmurny / deszczowy); 23.07 (↓10.8 pp, pochmurny / deszczowy); 24.07 (↓55.0 pp, pochmurny / deszczowy)
- **Hybryda szkodzi (12:00):** 20.07 (↑13.2 pp, słoneczny / mało chmur); 22.07 (↑11.2 pp, pochmurny / deszczowy); 25.07 (↑16.7 pp, mieszany); 28.07 (↑11.3 pp, mieszany); 29.07 (↑13.4 pp, mieszany)
- **Remis / szum (<1 pp):** 14.07, 15.07, 16.07, 17.07, 19.07, 26.07, 27.07, 30.07, 31.07, 01.08, 02.08, 03.08, 04.08, 05.08, 06.08, 07.08, 08.08, 09.08, 10.08, 11.08, 12.08, 13.08, 14.08, 15.08, 16.08, 17.08, 18.08, 19.08, 20.08, 21.08, 22.08, 23.08, 24.08

- **O 5:00:** hybryda ≈ raw (pomaga 1 dni / szkodzi 1) — przed wschodem prawie nie ma FoxESS do podmiany.

**Reguła operacyjna (z tych closeoutów):**

- Hybryda **najczęściej pomaga**, gdy poranek modelu był **zawyżony** (typowo dni **słabe / pochmurne** — u nas dominanta wśród „pomaga”: **pochmurny / deszczowy**): FoxESS „ściąga” sumę w dół.
- Hybryda **szkodzi**, gdy raw był **za niski** na jasny dzień (u nas dominanta wśród „szkodzi”: **mieszany**), a KPI brało ścieżkę hybrydową zanim dzień się domknął — stąd reguła **outlook = model_raw** do późnego dnia.
- **Wniosek:** hybryda godzinowa jest OK do sugestii urządzeń; **suma dnia do oceny modelu** = raw (albo hybryda dopiero wieczorem).

#### MAPE po retreningach / wdrożeniach

Podział według **zmian logiki / targetu / cech** (nie każdy niedzielny odśwież wag). Weekly retreningi wchodzą w erę dual od 27.07 (do ostatniego closeoutu). Szczegóły: `docs/NOTATKA_RETRENINGI_LIPIEC_2026.md`.

| Okres closeoutów | Retraining / wdrożenie | n | MAPE raw 5:00 | MAPE raw 12:00 |
|---|---|---:|---:|---:|
| 14.07–18.07 | przed targetem PVE (skala mieszana; GPS/ICON 17.07) | 5 | 22.8% | 20.6% |
| 19.07–26.07 | po PVE 18.07 ~16:32 — przed dual 26.07 | 8 | 26.5% | 21.7% |
| 27.07–24.08 | era produkcyjna dual (po 26.07; weekly = odświeżenie wag; do ostatniego closeoutu) | 29 | 11.0% | 10.0% |
| 19.07–24.08 | **era PVE łącznie** (bez 14–18) | 37 | **14.4%** | **12.6%** |

_Zakres całość: 14.07–24.08 (42 closeoutów) · MAPE raw 5:00 = **15.4%** · MAPE raw 12:00 = **13.5%**._


...

Najlepszy wariant dnia: Shadow XGB+TS 5:00 (APE 6.5%)

=== 2026-08-10 — closeout (primary + shadow) ===


| wariant | kWh | APE_% |
| --- | --- | --- |
| Actual (FoxESS app) | 28.900 | 0.000 |
| Primary RF 5:00 | 30.570 | 5.780 |
| Primary RF 12:00 | 26.060 | 9.830 |
| Shadow CS4 5:00 | 29.630 | 2.530 |
| Shadow CS4 12:00 | 26.000 | 10.030 |
| Shadow XGB+TS 5:00 | 30.430 | 5.290 |
| Shadow XGB+TS 12:00 | 24.190 | 16.300 |

Najlepszy wariant dnia: Shadow CS4 5:00 (APE 2.5%)

=== Shadow vs primary RF 16 ===


| porównanie | shadow_lepiej | primary_lepiej | pct_shadow |
| --- | --- | --- | --- |
| Live daily 5:00 (27.07→24.08) | 9/29 | 20/29 | 33.300 |
| Live midday 12:00 | 7/27 | 20/27 | 25.900 |
| XGB+TS daily (log 26.07–01.08) | 1/3 | 2/3 | 33.300 |
| Oneshot CS4 offline (19–25.07) | 5/7 | 2/7 | 71.400 |

CS4 daily wygrał: 06.08, 07.08, 08.08, 10.08, 16.08, 17.08, 18.08, 20.08, 21.08
Wniosek: primary RF 16 zostaje — shadow tylko obserwacja (launchd).


---
## Blok na zajęcia — diagnostyka źródeł danych

**Kontekst (2026-07-17):** tydzień pracy nad jakością wejść do modelu — nie nowy algorytm, tylko **uczciwsze dane i walidacja**.

| Temat | Decyzja |
|-------|----------|
| GPS instalacji | Observatorium → **dach** |
| Baseline fizyczny | stała 0,17 → **yield z train** |
| Model pogody OM | `best_match` → **`icon_seamless`** |
| Gate MLOps | `compare_model_change` → **ACCEPT** |

> Na obronie: to pokazuje, że błędy pochmurnych dni diagnozujemy **systemowo**, a nie „dokładamy cechy na ślepo”.

**Źródła / skrypty (blok diagnostyki):**
- GPS + ICON: [`docs/UPDATE_2026-07-17_gps-icon.md`](../docs/UPDATE_2026-07-17_gps-icon.md)
- gate zmian modelu: [`scripts/analysis/compare_model_change.py`](../scripts/analysis/compare_model_change.py)
- refetch pogody: [`scripts/analysis/fetch_weather.py`](../scripts/analysis/fetch_weather.py)


In [9]:
# Slajd 8c — tabele z metadata + forecast_validation.csv (dynamicznie)
import json

META_PREV = {
    'RF16': {'test_mae': 0.631, 'daily_mae': 3.818, 'gap': 0.076},
    'CS4': {'test_mae': 0.632, 'daily_mae': 3.847, 'gap': 0.080},
    'XGB+TS': {'test_mae': 0.625, 'daily_mae': 3.67, 'gap': 0.098},
}
META_FILES = {
    'RF16': ROOT / 'models' / 'pv_hourly_model.metadata.json',
    'CS4': ROOT / 'models' / 'pv_hourly_model_cs4.metadata.json',
    'XGB+TS': ROOT / 'models' / 'pv_hourly_model_xgb_ts.metadata.json',
}

retrain_rows = []
for name, path in META_FILES.items():
    m = json.loads(path.read_text(encoding='utf-8'))
    met = m['metrics']
    prev = META_PREV[name]
    retrain_rows.append({
        'model': name,
        'saved_at': m.get('saved_at', '')[:10],
        'train_end': m.get('train_end', '—'),
        'test_MAE': met['test_mae'],
        'daily_MAE': met['daily_mae'],
        'gap': met['gap'],
        'Δ_test_vs_02.08': round(met['test_mae'] - prev['test_mae'], 3),
    })

print('=== Retrening 09.08 — metryki offline ===')
show_table(pd.DataFrame(retrain_rows).round(3))

val = pd.read_csv(FORECASTS / 'forecast_validation.csv')
val['target_day'] = pd.to_datetime(val['target_day'])
week = val[(val['target_day'] >= '2026-08-03') & (val['target_day'] <= '2026-08-10')].copy()
for c in ['predicted_daily_raw', 'predicted_midday_raw', 'predicted_daily_cs4']:
    week[f'ape_{c}'] = (week[c] - week['actual_pv_report']).abs() / week['actual_pv_report'] * 100

week_tbl = pd.DataFrame({
    'dzień': week['target_day'].dt.strftime('%d.%m'),
    'actual_kWh': week['actual_pv_report'].round(1),
    'raw_5:00': week['predicted_daily_raw'].round(2),
    'APE_raw_%': week['ape_predicted_daily_raw'].round(1),
    'CS4_5:00': week['predicted_daily_cs4'].round(2),
    'APE_CS4_%': week['ape_predicted_daily_cs4'].round(1),
})
print('\n=== Tydzień 03–10.08 — closeout daily ===')
show_table(week_tbl)

era = val[val['target_day'] >= '2026-07-27'].copy()
for c in ['predicted_daily_raw', 'predicted_midday_raw']:
    era[f'ape_{c}'] = (era[c] - era['actual_pv_report']).abs() / era['actual_pv_report'] * 100
era_end = era['target_day'].max().strftime('%d.%m')

summary = pd.DataFrame([
    {'okres': '03–10.08 (n=7)', 'MAPE_raw_5:00_%': week['ape_predicted_daily_raw'].mean(),
     'MAPE_raw_12:00_%': week['ape_predicted_midday_raw'].mean(),
     'MAPE_CS4_5:00_%': week['ape_predicted_daily_cs4'].mean()},
    {'okres': f'Era dual 27.07–{era_end} (n={len(era)})',
     'MAPE_raw_5:00_%': era['ape_predicted_daily_raw'].mean(),
     'MAPE_raw_12:00_%': era['ape_predicted_midday_raw'].mean(),
     'MAPE_CS4_5:00_%': (
         (era['predicted_daily_cs4'] - era['actual_pv_report']).abs()
         / era['actual_pv_report'] * 100
     ).mean()},
])
print('\n=== MAPE — tydzień vs era dual ===')
show_table(summary.round(1))

cs4_win = week['ape_predicted_daily_cs4'] < week['ape_predicted_daily_raw']
print(
    f"CS4 lepsze od raw 5:00: {int(cs4_win.sum())}/{len(week)} dni — "
    + ', '.join(week.loc[cs4_win, 'target_day'].dt.strftime('%d.%m').tolist())
)
print('Notatki operacyjne: weather_notes id 117 (retrain) · 118 (tydzień)')


=== Retrening 09.08 — metryki offline ===


| model | saved_at | train_end | test_MAE | daily_MAE | gap | Δ_test_vs_02.08 |
| --- | --- | --- | --- | --- | --- | --- |
| RF16 | 2026-08-23 | 2026-08-22 | 0.658 | 3.666 | 0.069 | 0.027 |
| CS4 | 2026-08-23 | 2026-08-22 | 0.664 | 3.748 | 0.075 | 0.032 |
| XGB+TS | 2026-08-23 | — | 0.618 | 3.399 | 0.064 | -0.007 |


=== Tydzień 03–10.08 — closeout daily ===


| dzień | actual_kWh | raw_5:00 | APE_raw_% | CS4_5:00 | APE_CS4_% |
| --- | --- | --- | --- | --- | --- |
| 03.08 | 33.500 | 33.280 | 0.700 | 32.470 | 3.100 |
| 04.08 | 34.600 | 33.900 | 2.000 | 33.020 | 4.600 |
| 05.08 | 34.900 | 34.470 | 1.200 | 33.720 | 3.400 |
| 06.08 | 33.200 | 34.120 | 2.800 | 33.310 | 0.300 |
| 07.08 | 13.600 | 15.350 | 12.900 | 13.810 | 1.500 |
| 08.08 | 25.400 | 30.890 | 21.600 | 30.470 | 20.000 |
| 09.08 | 37.700 | 33.840 | 10.200 | 33.470 | 11.200 |
| 10.08 | 28.900 | 30.570 | 5.800 | 29.630 | 2.500 |


=== MAPE — tydzień vs era dual ===


| okres | MAPE_raw_5:00_% | MAPE_raw_12:00_% | MAPE_CS4_5:00_% |
| --- | --- | --- | --- |
| 03–10.08 (n=7) | 7.100 | 7.700 | 5.800 |
| Era dual 27.07–24.08 (n=29) | 11.000 | 10.000 | 11.400 |

CS4 lepsze od raw 5:00: 4/8 dni — 06.08, 07.08, 08.08, 10.08
Notatki operacyjne: weather_notes id 117 (retrain) · 118 (tydzień)


---
## Slajd A — Współrzędne pogody (~10 km błędu)

**Było:** Open-Meteo na punkcie Kraków-Observatorium (`50.0647, 19.9450`).  
**Jest:** współrzędne **dachu** instalacji (dokładne GPS tylko w lokalnym `.env`; w docs: okolice Krakowa).

| | |
|--|--|
| Skutek | Inna komórka siatki → inne chmury / radiacja |
| Działanie | Refetch archiwum `fetch_weather.py` + retrening RF |
| Instalacja | 5,39 kWp (11 paneli), tilt 35°, azymut S, bateria 10,36 kWh |

**Teza:** model ML nie „naprawi” złej lokalizacji źródła pogody.

**Źródła / skrypty:**
- refetch Open-Meteo na GPS dach: [`scripts/analysis/fetch_weather.py`](../scripts/analysis/fetch_weather.py)
- gate po zmianie GPS: [`scripts/analysis/compare_model_change.py`](../scripts/analysis/compare_model_change.py)
- notatka: [`docs/UPDATE_2026-07-17_gps-icon.md`](../docs/UPDATE_2026-07-17_gps-icon.md)


---
## Slajd B — Baseline fizyczny (uczciwy punkt odniesienia)

Prosty model: `ŷ = radiacja × yield`.

| | Było | Jest |
|--|------|------|
| Yield | stała **0,17** („na oko”) | mediana / OLS `PV / radiacja` **z train** |
| MAE baseline (dzienny) | ~16 kWh | ~5,6 kWh (OOF) |
| „Poprawa RF vs baseline” | ~**74%** (artefakt) | ~**24%** (realistycznie) |

**Zasada analityki:** parametr baseline’u też dopasowujemy tylko na train — tak jak hiperparametry RF.

Skrypty: `final_cv_production_split.py` (+ godzinowy analog).

**Źródła / skrypty:**
- baseline fizyczny (dzienny): [`scripts/analysis/final_cv_production_split.py`](../scripts/analysis/final_cv_production_split.py)
- analog godzinowy: [`scripts/analysis/final_cv_production_split_hourly.py`](../scripts/analysis/final_cv_production_split_hourly.py)


---
## Slajd C — Zachmurzenie: `best_match` vs ICON

Dni z dużym zawyżeniem PV (niska produkcja w app FoxESS):

| Dzień | App PV | OM best_match cloud / rad | **ICON** cloud / rad |
|-------|--------|---------------------------|----------------------|
| **09.07** | 17,3 kWh | 53% / 6,24 | **86% / 4,29** |
| **12.07** | 7,0 kWh | 89% / 4,34 | **96% / 1,42** |

**Wniosek:** domyślny Open-Meteo „gładzi” chmury.  
**Wdrożenie:** `OPENMETEO_MODEL=icon_seamless` → refetch archiwum → retrening.

IMGW (Balice) — audyt lipiec → **slajd C2** (ZIP miesięczny publikowany z opóźnieniem).

**Źródła / skrypty:**
- porównanie źródeł chmur: [`scripts/analysis/compare_cloud_sources.py`](../scripts/analysis/compare_cloud_sources.py)
- refetch + model OM: [`scripts/analysis/fetch_weather.py`](../scripts/analysis/fetch_weather.py) (`OPENMETEO_MODEL=icon_seamless`)
- oneshot ICON vs UKMO (później): [`scripts/analysis/oneshot_icon_vs_ukmo_precip.py`](../scripts/analysis/oneshot_icon_vs_ukmo_precip.py)
- notatka: [`docs/UPDATE_2026-07-17_gps-icon.md`](../docs/UPDATE_2026-07-17_gps-icon.md)



---
## Slajd C2 — IMGW audyt lipiec 2026 (10.08)

**Źródło:** [danepubliczne.imgw.pl](https://danepubliczne.imgw.pl) — archiwa ZIP miesięczne (nie live API).

### Dostępność archiwów lipiec 2026

| Archiwum | Plik | Status | Rola w projekcie |
|----------|------|--------|------------------|
| Synop **dobowe** | `2026_07_s.zip` | ✅ 31 dni Balice | USL (h słońca), opady — audyt zachmurzenia |
| Synop **terminowe** | `2026_07_s.zip` | ✅ ~744 h Balice | NOG oktanty → porównanie godzinowe z ICON |
| Klimat **dobowe** | `2026_07_k.zip` | ❌ 404 (jeszcze brak) | pokrywa śnieżna → `imgw_daily` |

> **Pokrywa śnieżna (klimat `_k.zip`):** dotyczy okresu **zimowego** — w praktyce od **listopada** wstecz (najwcześniej sprawdzamy `2026_11_k.zip` ~XII 2026). Lipiec/sierpień = 0 cm — brak wpływu na flagi `snow_on_panels`.

### Werdykt audytu (PLAN T1 — gate bez retrainu)

| Metryka | Wynik |
|---------|--------|
| Oneshot lipiec ICON vs IMGW NOG | bias **+13,4 pp**, corr **0,638** (jak czerwiec +12,8 pp) |
| IMGW USL vs słabe dni PV | potwierdza: 12.07 **0,1 h**, 24.07 **0,5 h** słońca |
| Model produkcyjny OM | **`icon_seamless`** od 17.07 — audyt **potwierdza** (ICON bardziej pochmurny niż `best_match`) |
| Retrening po audycie | **NIE** — synop IMGW nie wchodzi do cech RF; weekly 09.08 świeży |

**Procedura:** `compare_cloud_sources.py` → oneshot Balice → werdykt w notatce operacyjnej. Retrening dopiero gdy: (a) bug skali OM, albo (b) nowe dane **klimat** `_k.zip` + sezon zimowy.

**Źródła / skrypty:**
- audyt dni: [`scripts/analysis/compare_cloud_sources.py`](../scripts/analysis/compare_cloud_sources.py)
- oneshot czerwiec (wzór): [`scripts/analysis/oneshot_icon_imgw_clouds_june2026.py`](../scripts/analysis/oneshot_icon_imgw_clouds_june2026.py)
- IMGW śnieg (zima): [`scripts/analysis/fetch_imgw_snow.py`](../scripts/analysis/fetch_imgw_snow.py)
- plan: [`docs/PLAN_T1_T2_LIPIEC_2026.md`](../docs/PLAN_T1_T2_LIPIEC_2026.md) § IMGW `2026_07_s.zip`



In [10]:
# Slajd C2 — tabele z audytu IMGW lipiec (CSV lokalne, bez ponownego pobierania ZIP)
from pathlib import Path

AUDIT_CSV = DATA / 'cloud_source_comparison_july2026.csv'
ONESHOT_CSV = DATA / 'oneshot_icon_vs_imgw_balice_202607_daily.csv'

if not AUDIT_CSV.exists():
    print('⚠️ Brak:', AUDIT_CSV)
    print('   PYTHONPATH=$PWD python scripts/analysis/compare_cloud_sources.py \\')
    print('     --days 2026-07-09,2026-07-12,2026-07-21,2026-07-23,2026-07-24 \\')
    print('     --out data/processed/cloud_source_comparison_july2026.csv')
else:
    audit = pd.read_csv(AUDIT_CSV)
    tbl = pd.DataFrame({
        'dzień': audit['day'],
        'PV_app_kWh': audit['pv_app_kwh'].round(1),
        'ICON_cloud_%': audit['om_icon_seamless_cloud'].round(0),
        'best_match_%': audit['om_best_match_cloud'].round(0),
        'Δ_ICON−BM_pp': (audit['om_icon_seamless_cloud'] - audit['om_best_match_cloud']).round(1),
        'IMGW_USL_h': audit['imgw_sunshine_h'].round(1),
        'IMGW_opad_mm': audit['imgw_precip_mm'],
    })
    print('=== compare_cloud_sources — trudne dni lipca ===')
    show_table(tbl)

if ONESHOT_CSV.exists():
    daily = pd.read_csv(ONESHOT_CSV)
    if 'err_pp' not in daily.columns and {'icon', 'imgw'}.issubset(daily.columns):
        daily['err_pp'] = daily['icon'] - daily['imgw']
    cols = [c for c in ['day', 'icon', 'imgw', 'err_pp'] if c in daily.columns]
    if cols:
        print('\n=== Oneshot lipiec — ICON vs IMGW Balice (średnie dzienne) ===')
        show_table(daily[cols].round(1).tail(8))
        if 'err_pp' in daily.columns:
            print(f"Średni bias ICON−IMGW: {daily['err_pp'].mean():+.1f} pp")

print('\nPokrywa śnieżna: fetch_imgw_snow.py — sezon zimowy, najwcześniej listopad (_k.zip).')



=== compare_cloud_sources — trudne dni lipca ===


| dzień | PV_app_kWh | ICON_cloud_% | best_match_% | Δ_ICON−BM_pp | IMGW_USL_h | IMGW_opad_mm |
| --- | --- | --- | --- | --- | --- | --- |
| 2026-07-09 | 17.300 | 86.000 | 53.000 | 33.100 | 2.200 | 1.600 |
| 2026-07-12 | 7.000 | 96.000 | 89.000 | 7.200 | 0.100 | 0.100 |
| 2026-07-21 | 18.800 | 86.000 | 57.000 | 28.900 | 2.600 | 0.100 |
| 2026-07-23 | 20.600 | 90.000 | 66.000 | 24.200 | 2.500 | 16.700 |
| 2026-07-24 | 10.700 | 89.000 | 74.000 | 14.500 | 0.500 | 0.800 |


=== Oneshot lipiec — ICON vs IMGW Balice (średnie dzienne) ===


| icon | imgw | err_pp |
| --- | --- | --- |
| 86.100 | 87.500 | -1.400 |
| 49.500 | 40.600 | 8.900 |
| 41.400 | 36.500 | 5.000 |
| 82.700 | 68.800 | 13.900 |
| 61.300 | 40.100 | 21.200 |
| 42.000 | 11.500 | 30.500 |
| 23.300 | 6.200 | 17.100 |
| 31.000 | 24.000 | 7.100 |

Średni bias ICON−IMGW: +13.4 pp

Pokrywa śnieżna: fetch_imgw_snow.py — sezon zimowy, najwcześniej listopad (_k.zip).


---
## Slajd D — Gate MLOps: ICON → PVE

Protokół: `scripts/compare_model_change.py` — ten sam split 80/20, tolerancja Test MAE **+0,02**.

| Krok | Zmiana | Test MAE | Decyzja |
|------|--------|----------|---------|
| ICON (2026-07-17) | `best_match` → `icon_seamless` (∫pvPower) | 0.682 → **0.666** | **ACCEPT** |
| **PVE (2026-07-18)** | target = Δ`PVEnergyTotal` (jak app) | **0.582** | milestone (spójność zmiennej) |
| **Expanding (artefakt)** | ta sama metoda 80/20, okno → **2026-08-10** | **0.605** | **ocena offline** |
| **`.joblib` (weekly 09.08)** | train_end **2026-08-08** | **0.624** | **PRODUKCJA teraz** |

Gate offline PVE vs stary model na danych PVE: REVIEW (+0.014) — wdrażamy, bo trening = closeout = app.

**Źródła / skrypty:**
- bramka MLOps (Test MAE ±0,02): [`scripts/analysis/compare_model_change.py`](../scripts/analysis/compare_model_change.py)
- retrening po ICON/PVE: [`scripts/train/train_hourly_model_tuning.py`](../scripts/train/train_hourly_model_tuning.py)
- notatki: [`docs/UPDATE_2026-07-17_gps-icon.md`](../docs/UPDATE_2026-07-17_gps-icon.md) · [`docs/UPDATE_2026-07-18_target-pve.md`](../docs/UPDATE_2026-07-18_target-pve.md)


In [11]:
# Metryki z compare_model_change.py / train_hourly_model_tuning.py (lipiec 2026)
# Tabela na zajęcia — ewolucja jakości danych / modelu (lipiec 2026)
evo = pd.DataFrame([
    ['1. Start (holdout docs)', 'Observatorium + best_match', '∫pvPower / mixed', '0.661', 'archiwum'],
    ['2. GPS dach', '~50°N, ~20°E + best_match', '∫pvPower', '0.652', 'ACCEPT GPS'],
    ['3. ICON', 'GPS + icon_seamless', '∫pvPower', '0.666', 'ACCEPT (vs 0.682 na ICON)'],
    ['4. Target PVE', 'GPS + ICON', 'ΔPVEnergyTotal (= app)', '0.582', 'milestone 2026-07-18'],
    ['5. Expanding window', 'GPS + ICON', 'ΔPVEnergyTotal (= app)', '0.605', 'offline → 2026-08-10'],
    ['6. Artefakt .joblib', 'GPS + ICON', 'ΔPVEnergyTotal (= app)', '0.624', 'PRODUKCJA → 2026-08-08'],
], columns=['Krok', 'Pogoda / GPS', 'Target ML', 'Test MAE', 'Status'])
print('=== Ewolucja (lipiec 2026) ===')
show_table(evo)
print('\nProdukcja teraz: expanding Test MAE 0.605 · gap 0.029 · Daily 3.49 · .joblib Test MAE 0.624 · gap 0.057 · Daily 3.96 · 16 cech · 80/20 po dniach')


=== Ewolucja (lipiec 2026) ===


| Krok | Pogoda / GPS | Target ML | Test MAE | Status |
| --- | --- | --- | --- | --- |
| 1. Start (holdout docs) | Observatorium + best_match | ∫pvPower / mixed | 0.661 | archiwum |
| 2. GPS dach | ~50°N, ~20°E + best_match | ∫pvPower | 0.652 | ACCEPT GPS |
| 3. ICON | GPS + icon_seamless | ∫pvPower | 0.666 | ACCEPT (vs 0.682 na ICON) |
| 4. Target PVE | GPS + ICON | ΔPVEnergyTotal (= app) | 0.582 | milestone 2026-07-18 |
| 5. Expanding window | GPS + ICON | ΔPVEnergyTotal (= app) | 0.605 | offline → 2026-08-10 |
| 6. Artefakt .joblib | GPS + ICON | ΔPVEnergyTotal (= app) | 0.624 | PRODUKCJA → 2026-08-08 |


Produkcja teraz: expanding Test MAE 0.605 · gap 0.029 · Daily 3.49 · .joblib Test MAE 0.624 · gap 0.057 · Daily 3.96 · 16 cech · 80/20 po dniach


---
## Slajd 9 — Wdrożenie (MLOps)

**Harmonogram automatyczny** na Macu (`launchd` = jak cron: o stałych godzinach sam odpala skrypty).

### Pipeline end-to-end (dane → API → app)

Kompletny flow ML w repozytorium (warstwy `src/` → `scripts/` → `mlops/` → `api/`):

```text
FoxESS API + Open-Meteo ICON
         │
         ▼
  mlops/sync_data.py  ──►  SQLite (data/energy_model.db)
         │                      │
         │                      ▼
         │         load_hourly_training_frame_extended()
         │         (src/features/pv_features_hourly_extended.py)
         │                      │
         ▼                      ▼
  data/raw/ (opcj.)     scripts/train/train_hourly_model_tuning.py
                               │
                               ├─► MLflow (pv-hourly-forecast)
                               ├─► models/pv_hourly_model.joblib
                               └─► models/pv_hourly_model.metadata.json
                                        │
         mlops/forecast_pv.py ◄────────┘
                    │
                    ▼
         api/main.py  (model ładowany raz w app.state)
                    │
                    ▼
         GET /api/v1/forecast/hourly  ──►  aplikacja mobilna (mobile/)
                    │
                    ▼
         wieczorny closeout → forecast_validation.csv (MAPE live)
```

Docker: `docker compose up db api` · demo app: README §6 · raport wyników: [`05_raport_wynikow.ipynb`](05_raport_wynikow.ipynb)

### Pipeline (operacyjny) — dual od 26.07

```text
[FoxESS API]              [Open-Meteo ICON]
      │                          │
      └────────────┬─────────────┘
                   ▼
         ① SYNC  →  SQLite
                   │
         ┌─────────┴─────────┐
         ▼                   ▼
   ②a RF 16 cech        ②b RF CS4 (19)
   (primary)            (dual / shadow)
         │                   │
         ▼                   ▼
   pv_forecast.csv      pv_forecast_cs4.csv
   daily/midday/peak    daily_cs4 / …
         │                   │
         └─────────┬─────────┘
                   ▼
         ③ ranking AGD / advisor ← **tylko z 16**
                   ▼
         ④ CLOSEOUT 22:42 → actual vs **16 + CS4**
```

| Krok | Kiedy | Co |
|------|-------|-----|
| **① Sync** | 5:00 / 12:00 / 16:00 / 22:42 | FoxESS + pogoda → baza |
| **②a RF 16** | przy każdej prognozie | **oficjalna** prognoza |
| **②b RF CS4** | te same godziny (`FORECAST_CS4_ENABLED=1`) | drugi tor do porównania |
| **④ Closeout** | **22:42** | ocena vs app — obie ścieżki |

| Godzina | Skrypt | Rola |
|---------|--------|------|
| **05:00** | `daily_workflow.sh` | sync + **16 + CS4** + advisor |
| **12:00** | `midday_forecast.sh` | **16 + CS4** |
| **16:00** | `peak_arrival.sh` | **16 + CS4** |
| **22:42** | `evening_closeout.sh` | walidacja **16 vs CS4 vs app** |
| **nd 05:00** | `train_dual_weekly.sh` | retrain **obu** modeli |

- Archiwum: `data/processed/forecasts/` · dual: `*_cs4`
- Sterowanie falownikiem: dry-run (`BATTERY_CONTROL_ENABLED=0`)

**Źródła / skrypty:**
- sync: [`mlops/sync_data.py`](../mlops/sync_data.py)
- prognoza RF 16: [`mlops/forecast_pv.py`](../mlops/forecast_pv.py) · [`mlops/daily_workflow.sh`](../mlops/daily_workflow.sh) · [`mlops/midday_forecast.sh`](../mlops/midday_forecast.sh) · [`mlops/peak_arrival.sh`](../mlops/peak_arrival.sh)
- shadow CS4 / XGB: [`mlops/forecast_cs4_shadow.sh`](../mlops/forecast_cs4_shadow.sh) · [`mlops/forecast_xgb_ts_shadow.sh`](../mlops/forecast_xgb_ts_shadow.sh)
- closeout: [`mlops/evening_closeout.py`](../mlops/evening_closeout.py)
- retrening niedziela: [`mlops/train_dual_weekly.sh`](../mlops/train_dual_weekly.sh)
- launchd: [`mlops/install_launchd.sh`](../mlops/install_launchd.sh) · `config/launchd/`


In [12]:
# Job list = mlops/*.sh · historia prognoz = data/processed/forecasts/
jobs = pd.DataFrame([
    ['05:00', 'daily_workflow.sh', 'sync + 16 + CS4 + advisor', '2026-07-15 (CS4 od 26.07)'],
    ['12:00', 'midday_forecast.sh', 'sync + 16 + CS4', '2026-07-14 (CS4 od 26.07)'],
    ['16:00', 'peak_arrival.sh', 'sync + 16 + CS4', '2026-07-16 (CS4 od 26.07)'],
    ['22:42', 'evening_closeout.sh', 'walidacja 16 + CS4 vs app', '2026-07-15'],
    ['nd 05:00', 'train_dual_weekly.sh', 'retrain 16 + CS4', '2026-07-26'],
], columns=['Godzina', 'Skrypt', 'Opis', 'Start'])
show_table(jobs)

hist_path = FORECASTS / 'forecast_history.csv'
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    print('\n=== Ostatnie prognozy (16 + CS4) ===')
    show_table(hist.tail(9)[['run_at', 'run_label', 'target_day', 'predicted_kwh']])


| Godzina | Skrypt | Opis | Start |
| --- | --- | --- | --- |
| 05:00 | daily_workflow.sh | sync + 16 + CS4 + advisor | 2026-07-15 (CS4 od 26.07) |
| 12:00 | midday_forecast.sh | sync + 16 + CS4 | 2026-07-14 (CS4 od 26.07) |
| 16:00 | peak_arrival.sh | sync + 16 + CS4 | 2026-07-16 (CS4 od 26.07) |
| 22:42 | evening_closeout.sh | walidacja 16 + CS4 vs app | 2026-07-15 |
| nd 05:00 | train_dual_weekly.sh | retrain 16 + CS4 | 2026-07-26 |


=== Ostatnie prognozy (16 + CS4) ===


| run_at | run_label | target_day | predicted_kwh |
| --- | --- | --- | --- |
| 2026-08-22T05:00:34 | daily | 2026-08-22 | 22.590 |
| 2026-08-22T05:00:47 | daily_cs4 | 2026-08-22 | 21.660 |
| 2026-08-22T05:00:49 | daily_xgb_ts | 2026-08-22 | 22.340 |
| 2026-08-22T12:00:35 | midday | 2026-08-22 | 26.230 |
| 2026-08-22T12:00:37 | midday_cs4 | 2026-08-22 | 24.320 |
| 2026-08-22T12:00:38 | midday_xgb_ts | 2026-08-22 | 25.240 |
| 2026-08-22T16:00:39 | peak | 2026-08-22 | 24.320 |
| 2026-08-22T16:00:40 | peak_cs4 | 2026-08-22 | 22.950 |
| 2026-08-22T16:00:42 | peak_xgb_ts | 2026-08-22 | 24.760 |

---
## Slajd 9b — Raport tygodnia: 16 vs CS4 (19)

**Pytanie:** czy warto dorzucić warstwy chmur + clearness (CS4), skoro oficjalnie zostaje 16 cech?

### Co mierzymy

| | |
|--|--|
| **Okres** | 19–25.07.2026 |
| **Metoda** | oneshot **shadow** (trening ≤18.07, raw, bez adjust) — *nie* pełny tydzień live dual |
| **Live dual** | od **26.07** w tych samych jobach launchd; primary nadal **16** |
| **Źródło** | [`docs/UPDATE_2026-07-26_cs4-dual.md`](../docs/UPDATE_2026-07-26_cs4-dual.md) · CSV `oneshot_cs4_week_20260719_25.csv` |

### Gate offline (26.07)

| | 16 | CS4 (19) |
|--|----|----------|
| Test MAE [kWh/h] | 0,623 | **0,621** |
| Daily MAE [kWh/d] | 4,03 | **3,94** |
| Werdykt | — | **ACCEPT** (nie promujemy do primary) |

### Tydzień — MAE dzienne vs app

| Wariant | MAE | bliżej app |
|---------|-----|------------|
| Baseline **16** | 5,23 | — |
| **CS4 19** | **4,54** | **5/7 dni** |
| Live daily 5:00 (16) | 4,71 | — |

**Największy zysk CS4:** 21.07 i 24.07 (słabsza produkcja / chmury).  
**Decyzja:** 16 = oficjalne; CS4 = dual na produkcji do closeoutów ≥7 dni.

**Źródła / skrypty:**
- raport: [`docs/UPDATE_2026-07-26_cs4-dual.md`](../docs/UPDATE_2026-07-26_cs4-dual.md)
- CSV tygodnia oneshot: `data/processed/oneshot_cs4_week_20260719_25.csv`
- trening CS4 / dual: [`mlops/train_dual_weekly.sh`](../mlops/train_dual_weekly.sh) · [`scripts/analysis/run_cs4_sunday.sh`](../scripts/analysis/run_cs4_sunday.sh)
- gate 16 vs CS4: [`scripts/analysis/compare_model_change.py`](../scripts/analysis/compare_model_change.py)


In [13]:
# CSV: oneshot CS4 — docs/UPDATE_2026-07-26_cs4-dual.md
# Tydzień shadow 19–25.07 — Baseline 16 vs CS4 19 (oneshot)
week = pd.read_csv(DATA / 'oneshot_cs4_week_20260719_25.csv')
tbl = week[['day', 'actual', 'pred_A baseline', 'pred_CS4', 'err_A baseline', 'err_CS4', 'live_daily']].copy()
tbl['|err| 16'] = tbl['err_A baseline'].abs().round(2)
tbl['|err| CS4'] = tbl['err_CS4'].abs().round(2)
tbl['CS4 lepiej'] = tbl['|err| CS4'] < tbl['|err| 16']
tbl = tbl.rename(columns={
    'day': 'Dzień', 'actual': 'App',
    'pred_A baseline': 'Pred 16', 'pred_CS4': 'Pred CS4',
    'live_daily': 'Live 5:00 (16)',
})
show_table(tbl[['Dzień', 'App', 'Pred 16', 'Pred CS4', '|err| 16', '|err| CS4', 'CS4 lepiej', 'Live 5:00 (16)']])

mae16 = tbl['|err| 16'].mean()
mae_cs4 = tbl['|err| CS4'].mean()
print(f"MAE 16: {mae16:.2f}  |  MAE CS4: {mae_cs4:.2f}  |  CS4 bliżej app: {tbl['CS4 lepiej'].sum()}/7")
print('Raport:', DOCS / 'UPDATE_2026-07-26_cs4-dual.md')


| Dzień | App | Pred 16 | Pred CS4 | |err| 16 | |err| CS4 | CS4 lepiej | Live 5:00 (16) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-07-19 | 27.300 | 26.390 | 26.030 | 0.910 | 1.270 | False | 26.890 |
| 2026-07-20 | 37.400 | 31.170 | 31.460 | 6.230 | 5.940 | True | 33.760 |
| 2026-07-21 | 18.800 | 25.420 | 23.690 | 6.620 | 4.890 | True | 27.550 |
| 2026-07-22 | 33.500 | 28.370 | 27.780 | 5.130 | 5.720 | False | 29.150 |
| 2026-07-23 | 19.300 | 23.500 | 23.110 | 4.200 | 3.810 | True | 21.020 |
| 2026-07-24 | 7.700 | 18.650 | 16.440 | 10.950 | 8.740 | True | 21.480 |
| 2026-07-25 | 31.400 | 28.860 | 29.970 | 2.540 | 1.430 | True | 31.080 |

MAE 16: 5.23  |  MAE CS4: 4.54  |  CS4 bliżej app: 5/7
Raport: ./docs/UPDATE_2026-07-26_cs4-dual.md


---
## Slajd 9c — Raport tygodnia live dual: 26.07–01.08

**Pytanie:** po tygodniu **live** (16 + CS4 + od ~30.07 XGB+TS) — czy cokolwiek bije primary 16?

### Co mierzymy

| | |
|--|--|
| **Okres** | **26.07–01.08.2026** (7 closeoutów) |
| **Metoda** | **live** launchd — `forecast_validation.csv` (raw 5:00) + shadow CS4 / XGB+TS |
| **Primary** | RF **16** (oficjalne) |
| **Shadow** | CS4 **19** (od 27.07 w closeoucie); XGB+TS **24** (pełne daily od ~30.07) |
| **Źródło** | CSV `live_dual_week_20260726_0801.csv` |

### Agregat — MAE dzienny raw 5:00 vs app

| Wariant | n dni | MAE [kWh/d] | bliżej app niż 16 |
|---------|------:|------------:|-------------------|
| Primary **16** | 7 | **3,54** | — |
| Shadow **CS4** | 6 | 4,81 | **0/6** |
| Shadow **XGB+TS** | 3 | 4,52 | 1/3 (30.07) |

**Midday raw 12:00 (16):** MAE **3,12** — zwykle bliżej niż poranek na dniach z niedoszacowaniem NWP (np. 01.08: 26,6 → 31,1 przy actual 33,1).

### Co widać w tygodniu

| Wzorzec | Dni | Komentarz |
|---------|-----|-----------|
| **16 bardzo blisko** | 26.07, 27.07, 30.07 | słaby/burzowy 27.07 (~2,7% APE); 30.07 raw ≈ actual |
| **ICON za chmurny → niedoszac.** | 28–29.07, 31.07, 01.08 | Accu/MB jaśniej; CS4 **nie** pomaga (gorszy MAE) |
| **XGB+TS** | od 30.07 | za krótko na werdykt; 01.08 midday XGB ~32 kWh (dobrze), daily rano gorzej |
| **Alert IMGW 01/02.08** | wieczór deszcz ~20:30, burza ~północ | **nie psuje** PV 01.08 (po produkcji); słaby dzień **02.08** = skutek frontu |

### Decyzja (bez zmiany primary)

**Zostaje RF 16.** CS4 na tym tygodniu **nie** potwierdza zysku z oneshotu 19–25.07 — wspólny problem to NWP (ICON), nie brak warstw chmur. XGB+TS dalej w shadow. Korekta ADJUST nadal **OFF**.

**Źródła / skrypty:**
- closeouty live: [`mlops/evening_closeout.py`](../mlops/evening_closeout.py) → `forecast_validation.csv`
- CSV tygodnia: `data/processed/live_dual_week_20260726_0801.csv`
- dual shadow: [`mlops/forecast_cs4_shadow.sh`](../mlops/forecast_cs4_shadow.sh) · [`mlops/forecast_xgb_ts_shadow.sh`](../mlops/forecast_xgb_ts_shadow.sh)
- wykresy operacyjne: [`scripts/plots/plot_july_validation.py`](../scripts/plots/plot_july_validation.py) · [`scripts/plots/plot_production_validation.py`](../scripts/plots/plot_production_validation.py)


In [14]:
# CSV: live_dual_week ← forecast_validation.csv (evening_closeout.py)
# Live dual 26.07–01.08 — primary 16 vs CS4 vs XGB+TS (raw 5:00)
week = pd.read_csv(DATA / 'live_dual_week_20260726_0801.csv')
tbl = week.copy()
tbl['|err| 16'] = tbl['err_16'].abs().round(2)
tbl['|err| CS4'] = tbl['err_cs4'].abs().round(2)
tbl['|err| XGB'] = tbl['err_xgb'].abs().round(2)
tbl['CS4 lepiej'] = tbl['|err| CS4'] < tbl['|err| 16']
tbl['XGB lepiej'] = tbl['|err| XGB'] < tbl['|err| 16']
tbl = tbl.rename(columns={
    'day': 'Dzień', 'actual': 'App',
    'pred_16_daily': 'Pred 16', 'pred_cs4_daily': 'Pred CS4', 'pred_xgb_daily': 'Pred XGB',
})
show_table(tbl[['Dzień', 'App', 'Pred 16', 'Pred CS4', 'Pred XGB', '|err| 16', '|err| CS4', '|err| XGB', 'CS4 lepiej', 'XGB lepiej', 'note']])

mae16 = tbl['|err| 16'].mean()
mae_cs4 = tbl['|err| CS4'].mean(skipna=True)
mae_xgb = tbl['|err| XGB'].mean(skipna=True)
n_cs4 = tbl['|err| CS4'].notna().sum()
n_xgb = tbl['|err| XGB'].notna().sum()
print(f"MAE 16 (7 dni): {mae16:.2f}")
print(f"MAE CS4 ({n_cs4} dni): {mae_cs4:.2f}  |  CS4 bliżej: {tbl['CS4 lepiej'].fillna(False).sum()}/{n_cs4}")
print(f"MAE XGB ({n_xgb} dni): {mae_xgb:.2f}  |  XGB bliżej: {tbl['XGB lepiej'].fillna(False).sum()}/{n_xgb}")
print('Wykresy: ../docs/images/ml/july_validation_plot.png · production_validation_plot.png (do 24.08)')


| Dzień | App | Pred 16 | Pred CS4 | Pred XGB | |err| 16 | |err| CS4 | |err| XGB | CS4 lepiej | XGB lepiej | note |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2026-07-26 | 34.400 | 33.290 |  |  | 1.110 |  |  | False | False | start dual CS4 wieczorem / smoke |
| 2026-07-27 | 18.600 | 19.100 | 17.990 |  | 0.500 | 0.610 |  | False | False | słaby / burzowy — 16 bardzo blisko |
| 2026-07-28 | 34.600 | 28.330 | 27.240 |  | 6.270 | 7.360 |  | False | False | jasny — ICON niedoszac. ( Accu jaśniej) |
| 2026-07-29 | 36.500 | 31.310 | 30.100 |  | 5.190 | 6.400 |  | False | False | jasny — niedoszac. NWP |
| 2026-07-30 | 33.500 | 33.310 | 32.170 | 34.260 | 0.190 | 1.330 | 0.760 | False | False | bardzo dobrze — raw ≈ actual |
| 2026-07-31 | 34.700 | 29.650 | 28.550 | 30.160 | 5.050 | 6.150 | 4.540 | False | True | jasny — lekki niedoszac. |
| 2026-08-01 | 33.100 | 26.600 | 26.080 | 24.830 | 6.500 | 7.020 | 8.270 | False | False | dobry dzień PV; alert burzowy wieczór/noc → skutek na 02.08 |

MAE 16 (7 dni): 3.54
MAE CS4 (6 dni): 4.81  |  CS4 bliżej: 0/6
MAE XGB (3 dni): 4.52  |  XGB bliżej: 1/3
Wykresy: ../docs/images/ml/july_validation_plot.png · production_validation_plot.png (do 24.08)


---
## Slajd D8 — Korekta operacyjna (opcjonalnie, OFF w produkcji)

**Trzy warstwy — oceniamy osobno (nie mieszać w jednym dniu):**

| Warstwa | Co robi | Status |
|---------|---------|--------|
| **Raw RF 16** | Sam model na cały dzień | ✅ **produkcja** (CRON 5:00) |
| **CS4 shadow** | Inny RF (19 cech: low/mid cloud + clearness) | Obserwacja w closeoutach |
| **Adjust (D8)** | Skala intraday + chmury **nad** raw (bez retreningu) | **`FORECAST_OPERATIONAL_ADJUST=0`** |

**Reguła metodologiczna:** w jednym dniu porównujemy albo **16 vs CS4 vs app**, albo **raw vs adjust vs app** — nie oba eksperymenty naraz (inaczej nie wiadomo, co poprawiło wynik).

### Case study 21.07 (symulacja ręczna `manual_adjust_sim`)

| Wariant | Prognoza | App | Werdykt |
|---------|----------|-----|---------|
| Raw RF 16 | ~25,1 kWh | **18,8 kWh** | duże zawyżenie |
| **Adjust** | **~18,7 kWh** | **18,8 kWh** | trafił **D+0** |
| Adjust D+1/D+2 | wyżej niż raw | — | zawyżał kolejne dni → **nie włączać bez gate D8** |

**Decyzja D8 (plan T2):** ≥7 closeoutów raw vs symulowany adjust vs app → dopiero wtedy warunkowe `ADJUST=1` w `.env`.

**Źródła / skrypty:**
- moduł: [`src/models/intraday_forecast_adjust.py`](../src/models/intraday_forecast_adjust.py)
- symulacja: `FORECAST_OPERATIONAL_ADJUST=1 python scripts/forecast_pv.py --run-label manual_adjust_sim`
- plan: [`docs/PLAN_T1_T2_LIPIEC_2026.md`](../docs/PLAN_T1_T2_LIPIEC_2026.md) § D8
- docs: [`docs/UPDATE_2026-07-16_korekta-operacyjna.md`](../docs/UPDATE_2026-07-16_korekta-operacyjna.md)



In [15]:
# Slajd D8 — raw vs korekta operacyjna (intraday)
show_fig('intraday_raw_vs_adjust.png', width=900)

case = pd.DataFrame([
    {'wariant': 'Raw RF 16', 'prognoza_kWh': 25.1, 'app_kWh': 18.8, 'APE_%': 33.5},
    {'wariant': 'Adjust (symulacja 21.07)', 'prognoza_kWh': 18.7, 'app_kWh': 18.8, 'APE_%': 0.5},
])
print('=== Case study pochmurny 21.07 — tylko symulacja, nie CRON ===')
show_table(case)
print('Produkcja: FORECAST_OPERATIONAL_ADJUST=0 · CS4 = osobny shadow (slajd 9b/9c)')



<img src="../docs/images/ml/intraday_raw_vs_adjust.png" width="900" alt="intraday_raw_vs_adjust.png"/>

=== Case study pochmurny 21.07 — tylko symulacja, nie CRON ===


| wariant | prognoza_kWh | app_kWh | APE_% |
| --- | --- | --- | --- |
| Raw RF 16 | 25.100 | 18.800 | 33.500 |
| Adjust (symulacja 21.07) | 18.700 | 18.800 | 0.500 |

Produkcja: FORECAST_OPERATIONAL_ADJUST=0 · CS4 = osobny shadow (slajd 9b/9c)


---
## Slajd 10 — Green IT

**Świadome wybory:**
- **RF zamiast XGBoost** — Test MAE podobny (~0,60 vs ~0,61), ale **gap RF ~0,10 vs XGB ~0,47** (mniej przeuczenia, mniej iteracji tuningu)
- **16 cech zamiast 19** — mniejszy model, ten sam wynik
- **ICON tylko gdy potrzeba** — jeden model pogodowy, bez ensemble
- **Target = app** — zero post-processingu „dopasuj skalę” w treningu

**Źródła / skrypty:**
- porównanie RF vs XGB (gap): [`scripts/analysis/compare_algorithms_hourly.py`](../scripts/analysis/compare_algorithms_hourly.py)
- ablacja 16 vs 19: [`scripts/analysis/ablation_study.py`](../scripts/analysis/ablation_study.py)


In [16]:
# CSV: hourly_algorithm_comparison.csv ← compare_algorithms_hourly.py
cmp = pd.read_csv(DATA / 'hourly_algorithm_comparison.csv')
green = cmp[['label', 'test_mae_hour', 'gap_hour', 'verdict']].copy()
green['efficiency'] = (1 / green['test_mae_hour']).round(2)  # wyżej = lepiej na kWh błędu
green = green.sort_values('efficiency', ascending=False)
print('=== Green IT: dokładność vs overfit ===')
show_table(green.round(3))

print('\n=== Protokół zmian ML (compare_model_change) ===')
changelog = (ROOT / 'docs' / 'CHANGELOG_ML.md').read_text(encoding='utf-8')
import re
_doc_link = re.compile(r'\]\((UPDATE_[^)]+|PLAN_[^)]+|CHANGELOG_ML\.md|NOTATKA_[^)]+)\)')
changelog_nb = _doc_link.sub(r'](../docs/\1)', changelog)
print(changelog_nb[:1200] + '\n...')


=== Green IT: dokładność vs overfit ===


| label | test_mae_hour | gap_hour | verdict | efficiency |
| --- | --- | --- | --- | --- |
| RF (prod.) | 0.602 | 0.096 | ✅ Nie przeuczony | 1.660 |
| XGBoost | 0.614 | 0.470 | ❌ Przeuczony | 1.630 |
| Ridge | 0.831 | -0.003 | ✅ Nie przeuczony | 1.200 |


=== Protokół zmian ML (compare_model_change) ===
# Changelog ML — porównania zmian

Protokół: każda zmiana porównywana z baseline; **REJECT** gdy Test MAE >0,02 kWh/h lub operacyjny MAE +15%.

Generowanie: `python scripts/compare_model_change.py ... --append-changelog`

Reguły decyzji:

| Decyzja | Warunek |
|---------|---------|
| **ACCEPT** | Test MAE ≤ baseline + 0,02; brak regresji operacyjnej |
| **REVIEW** | Test MAE remis (+0,00…0,02); wymaga oceny operacyjnej (≥7 dni) |
| **REJECT** | Test MAE > baseline + 0,02 lub operacyjny MAE wyraźnie gorszy |

Metryki offline: ten sam split 80/20 po dniach (`random_state=42`).  
Metryki operacyjne: `forecast_validation.csv` (rolling N dni).

---

## Target godzinowy = PVEnergyTotal (jak w app) — 2026-07-18

**Raport wdrożeniowy:** [`UPDATE_2026-07-18_target-pve.md`](../docs/UPDATE_2026-07-18_target-pve.md) · wycofana skala: [`UPDATE_2026-07-18_skala-app.md`](../docs/UPDATE_2026-07-18_skala-app.md)

**Przyczyna:** trening na ∫`pvPower` (~w

---
## Slajd 11 — Antywzorce ML (czego unikać)

Typowe pułapki w projektach ML — **i jak ten projekt im zapobiega** (regresja PV kWh/h, nie klasyfikacja).

| Pułapka | Zły wzorzec | Nasze rozwiązanie |
|---------|-------------|-------------------|
| **Data leakage** | Rachunki Tauron / import z sieci w cechach PV | Target = Δ`PVEnergyTotal`; Tauron **tylko** do ROI — [`01_EDA` §7](../docs/01_EDA_analiza.md) |
| **Brak baseline** | Od razu RF/XGB bez punktu odniesienia | Baseline fizyczny (rad×yield z train) + Ridge + ablacja 1→16 cech |
| **Overfitting** | Metryki tylko na trainie (XGB gap **0,47**) | Holdout 80/20 po dniach; werdykt **gap + MAE** → RF 16 w produkcji |
| **Brak reprodukowalności** | Brak `random_state`, luźne wersje pakietów | `random_state=42`, pin w [`requirements.txt`](../requirements.txt) |
| **Hardcoded paths** | `pd.read_csv('/Users/.../dane.csv')` | `ROOT` + ścieżki względem repo — notebooki 01–05 |
| **Brak wdrożenia** | Tylko Jupyter, brak API | FastAPI · Docker · MLOps launchd · app mobilna |
| **Za dużo kodu w notebook** | 1000+ linii logiki w `.ipynb` | `src/` + `scripts/`; notebook = slajdy + demo |
| **Brak analizy błędów** | „MAE OK, koniec” | MAPE live, dni burzowe, typ chmur — slajd 8b |

**Specyficzne dla PV (pułapki, które omijamy):**
- **`pvPower` vs PVE** — inna skala niż app → target PVE od 18.07
- **KPI hybryda w trakcie dnia** — ocena modelu na **raw**, nie hybrydzie (reguła outlook)
- **Stała yield 0,17** — błędna skala baseline (archiwum; teraz yield z train)

**Źródła / skrypty:**
- decyzje: [`docs/03_ZALOZENIA_I_DECYZJE.md`](../docs/03_ZALOZENIA_I_DECYZJE.md)
- checklist: [`docs/PLAN_DYPLOM_CHECKLIST.md`](../docs/PLAN_DYPLOM_CHECKLIST.md)


In [17]:
# Antywzorce — skrót do Run All / Q&A
errors = [
    {
        'Pułapka': 'Data leakage',
        'Zły wzorzec': 'Tauron / import z sieci w cechach PV',
        'U nas': 'Target PVE; Tauron tylko ROI (docs/01_EDA §7)',
    },
    {
        'Pułapka': 'Brak baseline',
        'Zły wzorzec': 'Od razu RF/XGB bez porównania',
        'U nas': 'rad×yield + Ridge + ablacja 1→16',
    },
    {
        'Pułapka': 'Overfitting',
        'Zły wzorzec': 'Metryki tylko na train (XGB gap 0,47)',
        'U nas': 'Holdout po dniach; gap + MAE → RF 16',
    },
    {
        'Pułapka': 'Brak reprodukowalności',
        'Zły wzorzec': 'Brak random_state, luźne pakiety',
        'U nas': 'random_state=42 + pin requirements.txt',
    },
    {
        'Pułapka': 'Hardcoded paths',
        'Zły wzorzec': 'Ścieżki absolutne do Desktop',
        'U nas': 'ROOT + ścieżki względem repo',
    },
    {
        'Pułapka': 'Brak wdrożenia',
        'Zły wzorzec': 'Tylko notebook',
        'U nas': 'FastAPI + Docker + MLOps + mobile',
    },
    {
        'Pułapka': 'Za dużo kodu w notebook',
        'Zły wzorzec': 'Cała logika w .ipynb',
        'U nas': 'src/ + scripts/; notebook = demo',
    },
    {
        'Pułapka': 'Brak analizy błędów',
        'Zły wzorzec': 'Jedna metryka i koniec',
        'U nas': 'MAPE live, burze, chmury (slajd 8b)',
    },
    {
        'Pułapka': 'Skala app (PV)',
        'Zły wzorzec': 'pvPower / stała yield 0,17',
        'U nas': 'ΔPVEnergyTotal; yield kalibrowany z train',
    },
]

show_table(pd.DataFrame(errors))


| Pułapka | Zły wzorzec | U nas |
| --- | --- | --- |
| Data leakage | Tauron / import z sieci w cechach PV | Target PVE; Tauron tylko ROI (docs/01_EDA §7) |
| Brak baseline | Od razu RF/XGB bez porównania | rad×yield + Ridge + ablacja 1→16 |
| Overfitting | Metryki tylko na train (XGB gap 0,47) | Holdout po dniach; gap + MAE → RF 16 |
| Brak reprodukowalności | Brak random_state, luźne pakiety | random_state=42 + pin requirements.txt |
| Hardcoded paths | Ścieżki absolutne do Desktop | ROOT + ścieżki względem repo |
| Brak wdrożenia | Tylko notebook | FastAPI + Docker + MLOps + mobile |
| Za dużo kodu w notebook | Cała logika w .ipynb | src/ + scripts/; notebook = demo |
| Brak analizy błędów | Jedna metryka i koniec | MAPE live, burze, chmury (slajd 8b) |
| Skala app (PV) | pvPower / stała yield 0,17 | ΔPVEnergyTotal; yield kalibrowany z train |

---
## Slajd 12 — Wnioski

1. **Porównanie 3 modeli** — Ridge za słaby; XGB przeuczony (duży gap); **RF 16** = produkcja.
2. **Ta sama zmienna co app** — Δ`PVEnergyTotal` w treningu i closeoucie.
3. **ICON + GPS** — jakość wejść ważniejsza niż kolejny algorytm.
4. **Live (era dual 27.07–24.08)** — MAPE raw ~**11,0% / 10,0%** (n=29); **07.08** burza: CS4 **1,5%**; **10.08** pochmurny: CS4 **2,5%** vs raw **5,8%**; **17.08** front: CS4 **0,7%** (actual **21,5**); **23–24.08** jasne undershoot (~35 vs daily ~27–29); całość 14.07–24.08 ~**15,4%**.
5. **Tydzień 03–10.08 + retrening 09.08** (slajd **8c**) — upał OK; front **07–08** trudny; **10.08** CS4 wygrywa; weekly odświeżył wagi (**RF16 0,624**) bez podmiany prod.
6. **CS4 / XGB+TS shadow** — launchd obserwuje oba; live **CS4 4/15** daily lepiej od primary (era dual); **XGB+TS** czasem trafia lepiej w pojedynczy dzień (08.08 midday) — **zostaje RF 16** jako primary.
7. **MLOps z bramką** — ADJUST OFF (slajd **D8**); weekly = odświeżenie wag, nie zmiana logiki.

**Źródła / skrypty (podsumowanie ścieżki):**
- modelowanie: [`scripts/analysis/compare_algorithms_hourly.py`](../scripts/analysis/compare_algorithms_hourly.py) · [`scripts/train/train_hourly_model_tuning.py`](../scripts/train/train_hourly_model_tuning.py)
- live: [`mlops/evening_closeout.py`](../mlops/evening_closeout.py) · [`scripts/plots/plot_july_validation.py`](../scripts/plots/plot_july_validation.py)
- decyzje: [`docs/03_ZALOZENIA_I_DECYZJE.md`](../docs/03_ZALOZENIA_I_DECYZJE.md)


---
## Slajd 13 — Następne kroki (do obrony ~1 mies.)

**Najbliższe ~2 tygodnie:**
- zbierać **closeout dual** (16 vs CS4 vs XGB+TS vs app) — w tym złe / przejściowe dni
- notatki Accu / Meteoblue / UKMO na dni z deszczem (obserwacja, bez retrainu UKMO)
- **D8 adjust** — symulacja opisana na slajdzie **D8**; decyzja włączenia po obronie (gate ≥7 closeoutów)

**Error Analysis (TODO, T2+ / po obronie):** skala zachmurzenia ICON (0–1 vs 0–100) + feature importance w MLflow (chmury vs godzina/DOY) — [`docs/UPDATE_2026-08-02_error-analysis-cloud-fi.md`](../docs/UPDATE_2026-08-02_error-analysis-cloud-fi.md) · EA.1–EA.7 w PLAN

**Park:** geometria paneli · `pvlib` Ineichen · UKMO jako provider produkcji

**Świadomie poza scope:** auto-apply na falownik, drugi live provider pogody.

**Źródła / skrypty (backlog):**
- plan T1/T2: [`docs/PLAN_T1_T2_LIPIEC_2026.md`](../docs/PLAN_T1_T2_LIPIEC_2026.md)
- EA chmury/FI: [`docs/UPDATE_2026-08-02_error-analysis-cloud-fi.md`](../docs/UPDATE_2026-08-02_error-analysis-cloud-fi.md)
- bateria jesień/zima: [`docs/PLAN_BATERIA_JESIEN_ZIMA_2026.md`](../docs/PLAN_BATERIA_JESIEN_ZIMA_2026.md)


---
## Slajd 14 — Q&A

**GitHub:** [link do repo]  
**Demo:** `python scripts/forecast_pv.py --days 3 --sync`  
**Pytania?**

**Źródła / skrypty (na pytania):**
- decyzje: [`docs/03_ZALOZENIA_I_DECYZJE.md`](../docs/03_ZALOZENIA_I_DECYZJE.md)
- ML docs: [`docs/02_ML_predykcja_PV.md`](../docs/02_ML_predykcja_PV.md)
- changelog: [`docs/CHANGELOG_ML.md`](../docs/CHANGELOG_ML.md)
- porównanie algorytmów: [`scripts/analysis/compare_algorithms_hourly.py`](../scripts/analysis/compare_algorithms_hourly.py)
- closeout live: [`mlops/evening_closeout.py`](../mlops/evening_closeout.py)


In [18]:
# Checklist przed oddaniem (skrót)
checklist = {
    'Kod': ['Repo GitHub', 'README / docs/', 'requirements.txt', 'src/ + scripts/'],
    'Model': ['RF .joblib', 'GridSearch', '80/20 shuffle', 'compare_algorithms'],
    'Deploy': ['launchd / cron', 'forecast archive', 'battery dry-run'],
    'Dokumentacja': ['3 notebooki', 'CHANGELOG_ML', 'prezentacja'],
}
for cat, items in checklist.items():
    print(f'\n--- {cat} ---')
    for it in items:
        print(f'  [x] {it}')


--- Kod ---
  [x] Repo GitHub
  [x] README / docs/
  [x] requirements.txt
  [x] src/ + scripts/

--- Model ---
  [x] RF .joblib
  [x] GridSearch
  [x] 80/20 shuffle
  [x] compare_algorithms

--- Deploy ---
  [x] launchd / cron
  [x] forecast archive
  [x] battery dry-run

--- Dokumentacja ---
  [x] 3 notebooki
  [x] CHANGELOG_ML
  [x] prezentacja
